In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [2]:
def evaluate_classification_model(model, X_train, X_test, y_train, y_test, model_name='Model'):
    """Evaluate a classification model with various metrics"""
    # Train the model if not already trained
    if not hasattr(model, 'classes_'):
        model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    # Print classification report
    print(f"\n{model_name} Evaluation:")
    print(classification_report(y_test, y_pred))

    # Calculate ROC AUC
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    print(f"ROC AUC: {roc_auc:.3f}")

    # Return the evaluation metrics
    return {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'classification_report': classification_report(y_test, y_pred, output_dict=True),
        'confusion_matrix': confusion_matrix(y_test, y_pred),
        'roc_auc': roc_auc,
        'fpr': fpr,
        'tpr': tpr
    }


In [3]:
def cross_validate_model(model, X, y, cv=5, scoring='f1'):
    """Perform cross-validation and return scores"""
    skf = StratifiedKFold(n_splits=cv, shuffle=True, random_state=42)
    cv_scores = cross_val_score(model, X, y, cv=skf, scoring=scoring)

    print(f"\nCross-validation {scoring} scores: {cv_scores}")
    print(f"Mean {scoring} score: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    return cv_scores


In [4]:
def check_feature_correlation_with_target(X, y, feature_names):
    """Check correlation of features with target variable to detect data leakage"""
    correlations = []

    for i, feature in enumerate(feature_names):
        if isinstance(X, pd.DataFrame):
            corr = np.corrcoef(X[feature], y)[0, 1]
        else:
            corr = np.corrcoef(X[:, i], y)[0, 1]
        correlations.append((feature, abs(corr)))

    # Sort by absolute correlation
    correlations.sort(key=lambda x: x[1], reverse=True)

    print("\nFeature correlations with target:")
    for feature, corr in correlations:
        print(f"{feature}: {corr:.4f}")

    # Identify potentially leaking features
    high_corr_features = [feature for feature, corr in correlations if corr > 0.7]
    if high_corr_features:
        print(f"\nPotentially leaking features: {high_corr_features}")
    else:
        print("\nNo high correlation features found that would suggest leakage.")

    return correlations, high_corr_features

In [5]:
def estimate_business_impact(df, prediction_column, value_column, threshold=0.6):
    """Estimate business impact of churn predictions"""
    # Calculate at-risk revenue
    total_revenue = df[value_column].sum()
    at_risk_revenue = df[df[prediction_column] > threshold][value_column].sum()

    print(f"\nRevenue at risk (customers with >{threshold*100}% probability): ${at_risk_revenue:.2f}")
    print(f"Percentage of total revenue at risk: {at_risk_revenue/total_revenue*100:.1f}%")

    return {
        'total_revenue': total_revenue,
        'at_risk_revenue': at_risk_revenue,
        'percentage_at_risk': at_risk_revenue/total_revenue*100
    }